# AG3 - Actividad Guiada 3
## 03MIAR - Algoritmos de Optimización

**Nombre:** Donald Silva

**Link:** `<pegar aquí el enlace a tu notebook en Colab>`

**Github:** `https://github.com/dsilvas2001/03MIAR_DonaldSilva_AG_.git`

---

### Objetivos de la actividad

1. Desarrollar algoritmos con la técnica de **búsqueda aleatoria**
2. Desarrollar algoritmos con la técnica de **búsqueda local**
3. Desarrollar algoritmos con la técnica de **recocido simulado** (*simulated annealing*)
4. Desarrollar algoritmos con la técnica de **colonia de hormigas** *(no evaluable)*

### El problema

El **problema del viajante** (TSP): dadas $n$ ciudades y las distancias entre todas ellas, encontrar
el recorrido más corto que las visite todas exactamente una vez y regrese al punto de partida.

Se trabaja sobre la instancia **`swiss42`** de TSPLIB: 42 ciudades suizas con la matriz de
distancias dada explícitamente. Hay $41!/2 \approx 1{,}7 \times 10^{49}$ recorridos distintos, así
que enumerarlos está descartado: de ahí las metaheurísticas.

### Cómo se verifica este notebook

Toda solución que devuelve cualquiera de los cuatro algoritmos pasa por `es_solucion_valida()`, y
cada apartado termina con `assert` contra referencias comprobables. Si el notebook se ejecuta entero
sin excepciones, los algoritmos son correctos.

Solo usa la biblioteca estándar: no hace falta instalar nada.

---
## 1. Carga de los datos del problema

El enunciado advierte de que **la descarga puede fallar** y de que en ese caso hay que usar el
fichero `.tsp` adjunto. Para que el notebook arranque siempre, `cargar_problema()` prueba tres vías
en orden:

1. El fichero **`swiss42.tsp` local** (el adjunto; en Colab, súbelo con el panel de archivos).
2. La **descarga desde TSPLIB**.
3. Una **copia de la matriz incrustada** en este mismo notebook.

Todo lo que necesitan los algoritmos es preguntar la distancia entre dos ciudades. La clase
`ProblemaMatriz` ofrece `get_nodes()` y `get_weight(a, b)`, la misma interfaz que `tsplib95`, de
modo que **el resto del código es idéntico venga el dato de donde venga**.

El cargador entiende los dos formatos que aparecen en el enunciado: matriz explícita (`swiss42`) y
coordenadas, con lo que también sirven las instancias alternativas que el profesor deja comentadas,
`eil51` y `att48`.

In [1]:
import math
import random
import time

random.seed(42)   # Semilla fija: los experimentos son reproducibles


class ProblemaMatriz:
    """Instancia del TSP a partir de una matriz de distancias.

    Expone la misma interfaz que un objeto de tsplib95 (`get_nodes`, `get_weight`), así que
    los algoritmos no necesitan saber de dónde salieron los datos. Las ciudades se numeran
    siempre de 0 a n-1.
    """

    def __init__(self, matriz, nombre=""):
        self.matriz = matriz
        self.nombre = nombre

    def get_nodes(self):
        return list(range(len(self.matriz)))

    def get_weight(self, a, b):
        return self.matriz[a][b]


def parsear_tsp(texto):
    """Convierte el contenido de un fichero TSPLIB en una matriz de distancias.

    Admite las dos formas que aparecen en el enunciado:
      - EXPLICIT / FULL_MATRIX -> la matriz viene dada (swiss42)
      - NODE_COORD_SECTION     -> vienen coordenadas y hay que calcularla (eil51, att48)
    """
    cabecera = {}
    for linea in texto.splitlines():
        if ":" in linea and "SECTION" not in linea:
            clave, _, valor = linea.partition(":")
            cabecera[clave.strip().upper()] = valor.strip()

    if "EDGE_WEIGHT_SECTION" in texto:
        cuerpo = texto.split("EDGE_WEIGHT_SECTION")[1].replace("EOF", "")
        valores = [int(float(x)) for x in cuerpo.split()]
        n = int(cabecera["DIMENSION"])
        return [valores[i * n:(i + 1) * n] for i in range(n)]

    # Instancias dadas por coordenadas
    tipo = cabecera.get("EDGE_WEIGHT_TYPE", "EUC_2D").upper()
    cuerpo = texto.split("NODE_COORD_SECTION")[1].replace("EOF", "")

    puntos = []
    for linea in cuerpo.strip().splitlines():
        partes = linea.split()
        if len(partes) >= 3:
            puntos.append((float(partes[1]), float(partes[2])))

    n = len(puntos)
    matriz = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            dx = puntos[i][0] - puntos[j][0]
            dy = puntos[i][1] - puntos[j][1]
            if tipo == "ATT":                      # Distancia pseudo-euclídea (att48)
                r = math.sqrt((dx * dx + dy * dy) / 10.0)
                t = int(round(r))
                matriz[i][j] = t + 1 if t < r else t
            else:                                  # EUC_2D (eil51)
                matriz[i][j] = int(round(math.sqrt(dx * dx + dy * dy)))

    return matriz


def cargar_problema(instancia="swiss42", avisar=True):
    """Carga la instancia probando: fichero local -> descarga -> copia incrustada."""
    fichero = instancia + ".tsp"

    # 1) El fichero adjunto al enunciado
    try:
        with open(fichero, encoding="utf-8") as f:
            texto = f.read()
        if avisar:
            print(f"Datos leídos del fichero local '{fichero}'")
        return ProblemaMatriz(parsear_tsp(texto), instancia)
    except OSError:
        pass

    # 2) Descarga desde TSPLIB
    try:
        import gzip
        import urllib.request
        url = f"http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/{instancia}.tsp.gz"
        with urllib.request.urlopen(url, timeout=10) as respuesta:
            texto = gzip.decompress(respuesta.read()).decode("utf-8")
        with open(fichero, "w", encoding="utf-8") as f:
            f.write(texto)
        if avisar:
            print(f"Datos descargados de TSPLIB y guardados en '{fichero}'")
        return ProblemaMatriz(parsear_tsp(texto), instancia)
    except Exception as error:
        if avisar:
            print(f"La descarga ha fallado ({type(error).__name__}).")

    # 3) Copia incrustada en este notebook
    if instancia != "swiss42":
        raise RuntimeError(
            f"No hay copia incrustada de '{instancia}'. Sube el fichero {fichero} al notebook."
        )
    if avisar:
        print("Se usa la copia de la matriz incrustada en el notebook")
    matriz = [[int(v) for v in fila.split()] for fila in MATRIZ_SWISS42.strip().splitlines()]
    return ProblemaMatriz(matriz, instancia)


# Copia de la matriz de distancias de swiss42, como último recurso si no hay
# fichero local ni conexión. Es el contenido de EDGE_WEIGHT_SECTION del .tsp original.
MATRIZ_SWISS42 = """
    0 15 30 23 32 55 33 37 92 114 92 110 96 90 74 76 82 67 72 78 82 159 122 131 206 112 57 28 43 70 65 66 37 103 84 125 129 72 126 141 183 124
    15 0 34 23 27 40 19 32 93 117 88 100 87 75 63 67 71 69 62 63 96 164 132 131 212 106 44 33 51 77 75 72 52 118 99 132 132 67 139 148 186 122
    30 34 0 11 18 57 36 65 62 84 64 89 76 93 95 100 104 98 57 88 99 130 100 101 179 86 51 4 18 43 45 95 45 115 93 152 159 100 112 114 153 94
    23 23 11 0 11 48 26 54 70 94 69 89 75 84 84 89 92 89 54 78 99 141 111 109 190 89 44 11 29 54 56 89 47 118 96 147 151 90 122 126 163 101
    32 27 18 11 0 40 20 58 67 92 61 78 65 76 83 89 91 95 43 72 110 141 116 105 190 81 34 19 35 57 63 97 58 129 107 156 158 92 129 127 161 95
    55 40 57 48 40 0 23 55 96 123 78 75 62 36 56 66 63 95 37 34 137 174 156 129 224 90 15 59 75 96 103 105 91 158 139 164 156 78 169 163 191 115
    33 19 36 26 20 23 0 45 85 111 75 82 69 60 63 70 71 85 44 52 115 161 136 122 210 91 25 37 54 78 81 90 68 136 116 150 147 76 148 147 180 111
    37 32 65 54 58 55 45 0 124 149 118 126 113 80 42 42 49 40 87 60 94 195 158 163 242 135 65 63 79 106 101 50 66 118 104 109 103 36 160 178 218 153
    92 93 62 70 67 96 85 124 0 28 29 68 63 122 148 155 156 159 67 129 148 78 80 39 129 46 82 65 55 40 61 157 97 159 135 212 221 159 110 72 95 35
    114 117 84 94 92 123 111 149 28 0 54 91 88 150 174 181 182 181 95 157 159 50 65 27 102 65 110 87 73 50 68 176 112 166 142 229 241 184 99 46 69 38
    92 88 64 69 61 78 75 118 29 54 0 39 34 99 134 142 141 157 44 110 161 103 109 52 154 22 63 68 66 61 81 158 107 175 151 216 219 150 137 100 115 37
    110 100 89 89 78 75 82 126 68 91 39 0 14 80 129 139 135 167 39 98 187 136 148 81 186 28 61 92 97 98 117 173 134 204 181 232 229 153 176 137 143 62
    96 87 76 75 65 62 69 113 63 88 34 14 0 72 117 128 124 153 26 88 174 136 142 82 187 32 48 79 85 89 106 159 121 191 168 219 216 140 168 134 145 64
    90 75 93 84 76 36 60 80 122 150 99 80 72 0 59 71 63 116 56 25 170 201 189 151 252 104 44 95 111 130 138 130 127 192 174 186 172 90 205 193 214 135
    74 63 95 84 83 56 63 42 148 174 134 129 117 59 0 11 8 63 93 35 135 223 195 184 273 146 71 95 113 138 138 81 107 159 146 132 113 32 200 209 243 171
    76 67 100 89 89 66 70 42 155 181 142 139 128 71 11 0 11 54 103 46 130 230 198 192 279 155 80 99 117 143 141 74 107 155 143 122 102 22 202 215 250 179
    82 71 104 92 91 63 71 49 156 182 141 135 124 63 8 11 0 65 100 39 140 232 203 192 281 153 78 103 121 147 146 85 115 164 152 133 112 33 208 218 251 178
    67 69 98 89 95 95 85 40 159 181 157 167 153 116 63 54 65 0 127 92 83 224 180 199 269 175 106 95 109 135 125 21 80 107 100 71 63 33 173 205 249 191
    72 62 57 54 43 37 44 87 67 95 44 39 26 56 93 103 100 127 0 67 153 145 139 96 196 53 23 60 70 81 95 134 101 172 149 194 190 115 160 138 159 80
    78 63 88 78 72 34 52 60 129 157 110 98 88 25 35 46 39 92 67 0 152 207 188 162 258 119 48 89 107 129 134 108 114 176 159 163 147 66 200 197 224 147
    82 96 99 99 110 137 115 94 148 159 161 187 174 170 135 130 140 83 153 152 0 188 128 184 222 183 139 95 95 110 91 62 54 24 23 81 110 113 108 164 217 184
    159 164 130 141 141 174 161 195 78 50 103 136 136 201 223 230 232 224 145 207 188 0 65 57 51 109 160 132 116 90 102 217 148 188 168 264 281 231 100 26 30 75
    122 132 100 111 116 156 136 158 80 65 109 148 142 189 195 198 203 180 139 188 128 65 0 91 94 126 145 100 82 60 57 167 99 126 106 208 230 194 36 39 94 103
    131 131 101 109 105 129 122 163 39 27 52 81 82 151 184 192 192 199 96 162 184 57 91 0 106 53 115 104 94 74 94 196 134 192 168 251 260 197 126 64 64 19
    206 212 179 190 190 224 210 242 129 102 154 186 187 252 273 279 281 269 196 258 222 51 94 106 0 158 211 180 163 136 145 259 190 218 200 302 323 278 120 65 49 124
    112 106 86 89 81 90 91 135 46 65 22 28 32 104 146 155 153 175 53 119 183 109 126 53 158 0 75 89 88 83 103 178 129 197 173 236 238 166 156 111 115 34
    57 44 51 44 34 15 25 65 82 110 63 61 48 44 71 80 78 106 23 48 139 160 145 115 211 75 0 53 68 86 95 114 90 160 139 173 168 92 162 150 176 101
    28 33 4 11 19 59 37 63 65 87 68 92 79 95 95 99 103 95 60 89 95 132 100 104 180 89 53 0 18 44 45 92 42 112 89 149 156 99 111 116 155 97
    43 51 18 29 35 75 54 79 55 73 66 97 85 111 113 117 121 109 70 107 95 116 82 94 163 88 68 18 0 27 27 103 42 109 85 157 168 115 94 98 140 90
    70 77 43 54 57 96 78 106 40 50 61 98 89 130 138 143 147 135 81 129 110 90 60 74 136 83 86 44 27 0 21 128 62 119 96 179 192 142 79 72 115 74
    65 75 45 56 63 103 81 101 61 68 81 117 106 138 138 141 146 125 95 134 91 102 57 94 145 103 95 45 27 21 0 115 46 98 75 163 179 136 67 81 129 95
    66 72 95 89 97 105 90 50 157 176 158 173 159 130 81 74 85 21 134 108 62 217 167 196 259 178 114 92 103 128 115 0 69 86 81 60 65 54 158 195 243 190
    37 52 45 47 58 91 68 66 97 112 107 134 121 127 107 107 115 80 101 114 54 148 99 134 190 129 90 42 42 62 46 69 0 71 49 117 133 98 95 127 175 132
    103 118 115 118 129 158 136 118 159 166 175 204 191 192 159 155 164 107 172 176 24 188 126 192 218 197 160 112 109 119 98 86 71 0 24 94 127 137 100 163 218 194
    84 99 93 96 107 139 116 104 135 142 151 181 168 174 146 143 152 100 149 159 23 168 106 168 200 173 139 89 85 96 75 81 49 24 0 104 133 127 85 143 197 170
    125 132 152 147 156 164 150 109 212 229 216 232 219 186 132 122 133 71 194 163 81 264 208 251 302 236 173 149 157 179 163 60 117 94 104 0 39 100 190 241 292 246
    129 132 159 151 158 156 147 103 221 241 219 229 216 172 113 102 112 63 190 147 110 281 230 260 323 238 168 156 168 192 179 65 133 127 133 39 0 81 216 259 307 253
    72 67 100 90 92 78 76 36 159 184 150 153 140 90 32 22 33 33 115 66 113 231 194 197 278 166 92 99 115 142 136 54 98 137 127 100 81 0 193 214 253 187
    126 139 112 122 129 169 148 160 110 99 137 176 168 205 200 202 208 173 160 200 108 100 36 126 120 156 162 111 94 79 67 158 95 100 85 190 216 193 0 74 129 137
    141 148 114 126 127 163 147 178 72 46 100 137 134 193 209 215 218 205 138 197 164 26 39 64 65 111 150 116 98 72 81 195 127 163 143 241 259 214 74 0 55 80
    183 186 153 163 161 191 180 218 95 69 115 143 145 214 243 250 251 249 159 224 217 30 94 64 49 115 176 155 140 115 129 243 175 218 197 292 307 253 129 55 0 81
    124 122 94 101 95 115 111 153 35 38 37 62 64 135 171 179 178 191 80 147 184 75 103 19 124 34 101 97 90 74 95 190 132 194 170 246 253 187 137 80 81 0
"""

In [2]:
problem = cargar_problema("swiss42")

Nodos = list(problem.get_nodes())
N_CIUDADES = len(Nodos)

# --- Comprobación de que los datos son los que deben ser ---------------------------------------
assert N_CIUDADES == 42, f"Se esperaban 42 ciudades y hay {N_CIUDADES}"
assert problem.get_weight(0, 1) == 15, "La distancia 0-1 de swiss42 debe ser 15"
assert all(problem.get_weight(i, i) == 0 for i in Nodos), "La diagonal debe ser nula"
assert all(problem.get_weight(i, j) == problem.get_weight(j, i)
           for i in Nodos for j in Nodos), "La matriz debe ser simétrica"

print(f"\nInstancia '{problem.nombre}': {N_CIUDADES} ciudades")
print(f"Distancia entre las ciudades 0 y 1: {problem.get_weight(0, 1)}")
print(f"Recorridos posibles: {math.factorial(N_CIUDADES - 1) // 2:.3e}")

Datos leídos del fichero local 'swiss42.tsp'

Instancia 'swiss42': 42 ciudades
Distancia entre las ciudades 0 y 1: 15
Recorridos posibles: 1.673e+49


---
## 2. Funciones básicas

Una **solución** es una lista con las 42 ciudades en el orden en que se visitan, empezando siempre
por la 0. Al ser un ciclo, fijar la ciudad de partida no pierde ninguna solución y evita contar
42 veces el mismo recorrido.

`es_solucion_valida()` se añade a las funciones del enunciado como red de seguridad: **toda**
solución que devuelva cualquiera de los cuatro algoritmos pasa por ella.

In [3]:
def crear_solucion(Nodos):
    """Genera una solución aleatoria que empieza en el nodo 0."""
    resto = Nodos[1:]
    random.shuffle(resto)
    return [Nodos[0]] + resto


def distancia(a, b, problem):
    """Distancia entre dos ciudades."""
    return problem.get_weight(a, b)


def distancia_total(solucion, problem):
    """Longitud total del recorrido, incluido el regreso a la ciudad de partida."""
    total = 0
    for i in range(len(solucion) - 1):
        total += distancia(solucion[i], solucion[i + 1], problem)
    return total + distancia(solucion[len(solucion) - 1], solucion[0], problem)


def es_solucion_valida(solucion, problem):
    """Comprueba que el recorrido visita todas las ciudades una vez y empieza en la 0."""
    nodos = list(problem.get_nodes())
    return (len(solucion) == len(nodos)
            and solucion[0] == nodos[0]
            and sorted(solucion) == sorted(nodos))


# --- Verificación ------------------------------------------------------------------------------
sol_temporal = crear_solucion(Nodos)
assert es_solucion_valida(sol_temporal, problem)

# distancia_total contrastada con un cálculo alternativo del ciclo, con aritmética modular
alternativa = sum(problem.get_weight(sol_temporal[i], sol_temporal[(i + 1) % N_CIUDADES])
                  for i in range(N_CIUDADES))
assert distancia_total(sol_temporal, problem) == alternativa

# El recorrido 0,1,2,...,41 debe medir lo mismo recorrido al revés (la matriz es simétrica)
identidad = list(range(N_CIUDADES))
invertida = [0] + identidad[1:][::-1]
assert distancia_total(identidad, problem) == distancia_total(invertida, problem)

print("Verificado: distancia_total es coherente y las soluciones generadas son válidas.")
print(f"\nSolución aleatoria de ejemplo: {sol_temporal[:10]} ...")
print(f"Su distancia total: {distancia_total(sol_temporal, problem)}")

Verificado: distancia_total es coherente y las soluciones generadas son válidas.

Solución aleatoria de ejemplo: [0, 10, 4, 11, 37, 26, 25, 31, 12, 5] ...
Su distancia total: 4543


---
## 3. Búsqueda aleatoria

La técnica más simple: generar soluciones al azar y quedarse con la mejor. No usa ninguna
información del problema ni siquiera mira si una solución se parece a otra buena, así que sirve
de **referencia inferior**: cualquier otra técnica debe batirla con claridad o no está aportando
nada.

**Criterios de parada.** El enunciado se para al cabo de $N$ iteraciones y apunta que *«podemos
incluir otros»*. Se implementan tres, combinables, y se detiene con el primero que se cumpla:

| Criterio | Cuándo conviene |
|---|---|
| **Número de muestras** | Cuando se quiere un esfuerzo fijo y comparable entre ejecuciones |
| **K iteraciones sin mejora** | Cuando interesa parar al estancarse, sin fijar el esfuerzo de antemano |
| **Presupuesto de tiempo** | Cuando el límite real es el reloj y no el número de evaluaciones |

In [4]:
def busqueda_aleatoria(problem, N=None, sin_mejora=None, segundos=None):
    """Genera soluciones al azar y se queda con la mejor.

    Criterios de parada (hay que dar al menos uno; se para con el primero que se cumpla):
      - N          : número máximo de soluciones evaluadas
      - sin_mejora : parar tras esas iteraciones seguidas sin mejorar
      - segundos   : presupuesto de tiempo

    Devuelve (mejor_solucion, mejor_distancia, evaluadas).
    """
    if N is None and sin_mejora is None and segundos is None:
        raise ValueError("Hay que indicar al menos un criterio de parada")

    Nodos = list(problem.get_nodes())

    mejor_solucion = crear_solucion(Nodos)
    mejor_distancia = distancia_total(mejor_solucion, problem)

    inicio = time.perf_counter()
    evaluadas = 1
    desde_la_ultima_mejora = 0

    while True:
        if N is not None and evaluadas >= N:
            break
        if sin_mejora is not None and desde_la_ultima_mejora >= sin_mejora:
            break
        if segundos is not None and time.perf_counter() - inicio >= segundos:
            break

        solucion = crear_solucion(Nodos)                 # Genera una solución aleatoria
        d = distancia_total(solucion, problem)           # La evalúa
        evaluadas += 1

        if d < mejor_distancia:                          # Y la guarda si mejora
            mejor_solucion, mejor_distancia = solucion, d
            desde_la_ultima_mejora = 0
        else:
            desde_la_ultima_mejora += 1

    return mejor_solucion, mejor_distancia, evaluadas


random.seed(42)
solucion_aleatoria, distancia_aleatoria, evaluadas = busqueda_aleatoria(problem, N=10000)

assert es_solucion_valida(solucion_aleatoria, problem)
print("Mejor solución:", solucion_aleatoria)
print("Distancia     :", distancia_aleatoria, f"  ({evaluadas} soluciones evaluadas)")

Mejor solución: [0, 29, 28, 32, 20, 2, 10, 27, 40, 38, 39, 7, 17, 11, 30, 4, 3, 16, 19, 6, 18, 9, 21, 23, 22, 8, 24, 12, 26, 13, 37, 14, 36, 25, 41, 34, 33, 1, 35, 31, 5, 15]
Distancia     : 3624   (10000 soluciones evaluadas)


In [5]:
# Los tres criterios de parada, uno al lado de otro
print(f"{'criterio de parada':<34} | {'evaluadas':>10} | {'distancia':>10} | {'segundos':>9}")
print("-" * 74)

for etiqueta, argumentos in [
    ("N = 1 000 muestras",            {"N": 1000}),
    ("N = 10 000 muestras",           {"N": 10000}),
    ("2 000 iteraciones sin mejora",  {"sin_mejora": 2000}),
    ("1 segundo de reloj",            {"segundos": 1.0}),
]:
    random.seed(42)                                   # Mismo punto de partida en los cuatro
    inicio = time.perf_counter()
    _, d, evaluadas = busqueda_aleatoria(problem, **argumentos)
    print(f"{etiqueta:<34} | {evaluadas:>10,} | {d:>10} | {time.perf_counter() - inicio:>9.2f}")

print("\nMultiplicar por diez el esfuerzo apenas mejora el resultado: muestrear al azar en un")
print("espacio de 1,7e49 recorridos es, por sí solo, una estrategia muy pobre.")

criterio de parada                 |  evaluadas |  distancia |  segundos
--------------------------------------------------------------------------
N = 1 000 muestras                 |      1,000 |       3749 |      0.03
N = 10 000 muestras                |     10,000 |       3624 |      0.27
2 000 iteraciones sin mejora       |      3,204 |       3703 |      0.08
1 segundo de reloj                 |     36,398 |       3624 |      1.00

Multiplicar por diez el esfuerzo apenas mejora el resultado: muestrear al azar en un
espacio de 1,7e49 recorridos es, por sí solo, una estrategia muy pobre.


---
## 4. Búsqueda local

En lugar de saltar a soluciones sin relación entre sí, la búsqueda local **explora el entorno** de
la solución actual: genera sus *vecinas*, se mueve a la mejor y repite mientras haya mejora. Cuando
ninguna vecina mejora, se ha llegado a un **óptimo local** y se para.

### El operador de vecindad es la decisión de diseño

Lo que define esta técnica es qué se considera «vecina». El enunciado usa un operador y señala que
*«se puede modificar para aplicar otros generadores»*, así que se implementan los dos naturales
sobre un recorrido, y `genera_vecina` recibe cuál usar:

| Operador | Qué hace con las posiciones $i$ y $j$ | Efecto sobre el recorrido |
|---|---|---|
| **Intercambio** | Permuta las dos ciudades | Cambia **4 aristas** del ciclo |
| **Inversión de segmento (2-opt)** | Da la vuelta al tramo entre $i$ y $j$ | Cambia solo **2 aristas** |

Ambos generan $(n-1)(n-2)/2$ vecinas, o sea que **cuestan lo mismo**. La diferencia es la calidad
del movimiento: 2-opt sustituye dos aristas por otras dos y es exactamente el movimiento que
**deshace los cruces** del recorrido, que son la causa típica de que un ciclo sea largo. Por eso los
dos se comparan más abajo desde los mismos arranques, que es lo único que aísla el efecto del
operador.

Conviene no confundir los nombres: *2-opt* designa la inversión de segmento, no el intercambio de
dos ciudades.

In [6]:
def vecina_intercambio(solucion, i, j):
    """Intercambia las ciudades de las posiciones i y j."""
    vecina = solucion[:]
    vecina[i], vecina[j] = vecina[j], vecina[i]
    return vecina


def vecina_2opt(solucion, i, j):
    """Invierte el segmento comprendido entre las posiciones i y j (movimiento 2-opt)."""
    return solucion[:i] + solucion[i:j + 1][::-1] + solucion[j + 1:]


def genera_vecina(solucion, problem, operador=vecina_2opt):
    """Devuelve la mejor solución vecina y su distancia, según el operador indicado.

    Se recorren todos los pares de posiciones (i, j). La posición 0 queda fija porque
    la ciudad de partida no se mueve.
    """
    mejor_solucion, mejor_distancia = None, float("inf")

    for i in range(1, len(solucion) - 1):
        for j in range(i + 1, len(solucion)):
            vecina = operador(solucion, i, j)
            d = distancia_total(vecina, problem)
            if d < mejor_distancia:
                mejor_solucion, mejor_distancia = vecina, d

    return mejor_solucion, mejor_distancia


def busqueda_local(problem, operador=vecina_2opt, solucion_inicial=None):
    """Se mueve a la mejor vecina mientras haya mejora; para en el óptimo local.

    Devuelve siempre una solución válida, también cuando la primera vecina ya no mejora.
    """
    Nodos = list(problem.get_nodes())
    actual = solucion_inicial[:] if solucion_inicial else crear_solucion(Nodos)
    distancia_actual = distancia_total(actual, problem)

    iteraciones = 0
    while True:
        vecina, distancia_vecina = genera_vecina(actual, problem, operador)

        if distancia_vecina >= distancia_actual:      # No se puede mejorar: óptimo local
            return actual, distancia_actual, iteraciones

        actual, distancia_actual = vecina, distancia_vecina
        iteraciones += 1


random.seed(7)
inicial = crear_solucion(Nodos)
print("Distancia de la solución inicial:", distancia_total(inicial, problem))

sol_local, dist_local, iteraciones = busqueda_local(problem, vecina_2opt, inicial)
assert es_solucion_valida(sol_local, problem)
print(f"Tras {iteraciones} iteraciones de búsqueda local con 2-opt: {dist_local}")

Distancia de la solución inicial: 4485
Tras 36 iteraciones de búsqueda local con 2-opt: 1315


In [7]:
# Los dos operadores, desde LOS MISMOS arranques: es lo único que aísla su efecto
random.seed(7)
ARRANQUES = [crear_solucion(Nodos) for _ in range(5)]

print(f"{'arranque':>9} | {'inicial':>8} | {'intercambio':>12} | {'2-opt':>8}")
print("-" * 48)

resultados_intercambio, resultados_2opt = [], []
for k, inicio in enumerate(ARRANQUES):
    _, d_int, _ = busqueda_local(problem, vecina_intercambio, inicio)
    _, d_2opt, _ = busqueda_local(problem, vecina_2opt, inicio)
    resultados_intercambio.append(d_int)
    resultados_2opt.append(d_2opt)
    print(f"{k + 1:>9} | {distancia_total(inicio, problem):>8} | {d_int:>12} | {d_2opt:>8}")

media_int = sum(resultados_intercambio) / len(resultados_intercambio)
media_2opt = sum(resultados_2opt) / len(resultados_2opt)
print("-" * 48)
print(f"{'media':>9} | {'':>8} | {media_int:>12.0f} | {media_2opt:>8.0f}")

# Partiendo de lo mismo, la inversión de segmento gana siempre
assert min(resultados_2opt) < min(resultados_intercambio)
assert media_2opt < media_int

mejor_local = min(resultados_2opt)
print(f"\nMejor con inversión de segmento: {mejor_local}")
print(f"Mejor con intercambio          : {min(resultados_intercambio)}")
print("\nMismo coste por iteración y mismo número de vecinas: la diferencia está entera")
print("en la calidad del movimiento, no en el esfuerzo invertido.")

 arranque |  inicial |  intercambio |    2-opt
------------------------------------------------
        1 |     4485 |         1752 |     1315
        2 |     4638 |         1518 |     1301
        3 |     4898 |         1722 |     1331
        4 |     5342 |         1907 |     1373
        5 |     4768 |         1968 |     1335
------------------------------------------------
    media |          |         1773 |     1331

Mejor con inversión de segmento: 1301
Mejor con intercambio          : 1518

Mismo coste por iteración y mismo número de vecinas: la diferencia está entera
en la calidad del movimiento, no en el esfuerzo invertido.


---
## 5. Recocido simulado

La búsqueda local tiene un problema: se detiene en el primer óptimo local y no sabe salir de él. El
recocido simulado lo resuelve **aceptando a veces soluciones peores**, con probabilidad

$$P = e^{-\Delta / T}$$

donde $\Delta$ es cuánto empeora y $T$ la *temperatura*, que va bajando. Al principio, con $T$ alta,
se acepta casi cualquier cosa y la búsqueda se mueve con libertad; al enfriarse, solo se aceptan
empeoramientos pequeños y el proceso converge. Es la analogía con el enfriamiento lento de un metal.

**Dos decisiones de diseño:**

1. **Cómo se elige la vecina.** El enunciado la genera intercambiando dos ciudades al azar y apunta
   que es *«mejorable eligiendo otra forma de elegir una vecina»*. Se usa la **inversión de
   segmento**, coherente con lo que ha demostrado el apartado 4.

2. **La temperatura inicial.** Tiene que guardar relación con la magnitud de los $\Delta$ del
   problema: si es desmesurada, $e^{-\Delta/T} \approx 1$, se acepta todo y durante media ejecución
   el algoritmo no es más que un paseo aleatorio. Aquí se **calibra**, muestreando empeoramientos
   reales y despejando $T$ de $e^{-\Delta/T} = 0{,}8$, para arrancar aceptando en torno al 80 %.

In [8]:
def genera_vecina_aleatorio(solucion, operador=vecina_2opt):
    """Genera UNA vecina al azar, eligiendo dos posiciones y aplicando el operador."""
    i, j = sorted(random.sample(range(1, len(solucion)), 2))
    return operador(solucion, i, j)


def probabilidad(T, d):
    """¿Se acepta un empeoramiento de d a la temperatura T?"""
    return random.random() < math.exp(-d / T)


def bajar_temperatura(T, alfa=0.995):
    """Enfriamiento geométrico."""
    return T * alfa


def temperatura_inicial(problem, aceptacion=0.8, muestras=200):
    """Calibra T0 para aceptar al principio en torno a `aceptacion` de los empeoramientos.

    Se muestrean empeoramientos reales del problema y se despeja T de exp(-delta/T) = aceptacion.
    """
    Nodos = list(problem.get_nodes())
    solucion = crear_solucion(Nodos)
    d_actual = distancia_total(solucion, problem)

    empeoramientos = []
    for _ in range(muestras):
        vecina = genera_vecina_aleatorio(solucion)
        d_vecina = distancia_total(vecina, problem)
        if d_vecina > d_actual:
            empeoramientos.append(d_vecina - d_actual)
        solucion, d_actual = vecina, d_vecina

    if not empeoramientos:
        return 1.0

    delta_medio = sum(empeoramientos) / len(empeoramientos)
    return -delta_medio / math.log(aceptacion)


def recocido_simulado(problem, TEMPERATURA=None, alfa=0.995,
                      iteraciones_por_T=20, T_minima=1e-3, operador=vecina_2opt):
    """Recocido simulado sobre el TSP.

    Devuelve (mejor_solucion, mejor_distancia, evaluadas).
    """
    Nodos = list(problem.get_nodes())

    if TEMPERATURA is None:
        TEMPERATURA = temperatura_inicial(problem)

    solucion_referencia = crear_solucion(Nodos)
    distancia_referencia = distancia_total(solucion_referencia, problem)

    # La mejor de todas se guarda siempre, aunque la búsqueda se haya movido a otra peor
    mejor_solucion = solucion_referencia[:]
    mejor_distancia = distancia_referencia

    evaluadas = 0
    while TEMPERATURA > T_minima:
        # Varias vecinas por temperatura: da tiempo a explorar antes de enfriar
        for _ in range(iteraciones_por_T):
            vecina = genera_vecina_aleatorio(solucion_referencia, operador)
            distancia_vecina = distancia_total(vecina, problem)
            evaluadas += 1

            if distancia_vecina < mejor_distancia:
                mejor_solucion, mejor_distancia = vecina[:], distancia_vecina

            delta = distancia_vecina - distancia_referencia
            if delta < 0 or probabilidad(TEMPERATURA, delta):
                solucion_referencia, distancia_referencia = vecina, distancia_vecina

        TEMPERATURA = bajar_temperatura(TEMPERATURA, alfa)

    return mejor_solucion, mejor_distancia, evaluadas


random.seed(42)
T0 = temperatura_inicial(problem)
print(f"Temperatura de referencia según el calibrado: {T0:.1f}")
print("(el enunciado la fija en 10 000 000, con lo que exp(-delta/T) ~ 1 y al principio")
print(" se acepta absolutamente todo)")
print("recocido_simulado vuelve a calibrar por su cuenta al no recibir TEMPERATURA, así que")
print("arranca en una temperatura de este orden, no en este valor exacto.\n")

sol_recocido, dist_recocido, evaluadas = recocido_simulado(problem)
assert es_solucion_valida(sol_recocido, problem)
print("Mejor solución:", sol_recocido)
print(f"Distancia     : {dist_recocido}   ({evaluadas:,} soluciones evaluadas)")

Temperatura de referencia según el calibrado: 278.7
(el enunciado la fija en 10 000 000, con lo que exp(-delta/T) ~ 1 y al principio
 se acepta absolutamente todo)
recocido_simulado vuelve a calibrar por su cuenta al no recibir TEMPERATURA, así que
arranca en una temperatura de este orden, no en este valor exacto.

Mejor solución: [0, 1, 6, 4, 3, 27, 2, 28, 29, 30, 38, 22, 39, 21, 24, 40, 23, 41, 9, 8, 10, 25, 11, 12, 18, 26, 5, 13, 19, 14, 16, 15, 37, 7, 17, 31, 36, 35, 20, 33, 34, 32]
Distancia     : 1273   (50,840 soluciones evaluadas)


---
## 6. Colonia de hormigas *(no evaluable)*

Cada hormiga construye un recorrido ciudad a ciudad. Al terminar, deposita **feromona** en las
aristas que ha usado, en cantidad inversamente proporcional a la longitud de su recorrido: cuanto
mejor el trayecto, más rastro deja. La feromona se **evapora** con el tiempo, de modo que solo las
aristas reforzadas una y otra vez acaban destacando.

### Las tres mejoras que pide el enunciado

**1. La elección del siguiente nodo.** El enunciado la hace uniformemente al azar y advierte de que
*«para ser más eficiente debería seleccionar el próximo nodo siguiendo la probabilidad»*

$$p^k_{ij} = \frac{[\tau_{ij}]^\alpha [\nu_{ij}]^\beta}{\sum_{l \in J^k_i} [\tau_{il}]^\alpha [\nu_{il}]^\beta}$$

con $\tau_{ij}$ la feromona de la arista y $\nu_{ij} = 1/d_{ij}$ la **visibilidad** (cuanto más
cerca, más atractiva). Los exponentes gradúan a qué hacer caso: $\alpha$ pesa la experiencia
acumulada por la colonia y $\beta$ la avidez por lo cercano. Se implementa por **ruleta**.

Esta es la mejora decisiva: eligiendo al azar, la feromona se deposita y se evapora pero **no
influye en nada**, y el algoritmo no es más que búsqueda aleatoria con pasos extra.

**2. La evaporación.** El enunciado resta una cantidad fija. Se sustituye por la forma habitual,
**multiplicativa**, $\tau \leftarrow (1-\rho)\,\tau$ con un suelo mínimo: así la feromona decae en
proporción a lo que hay, en vez de arrasar por igual los rastros débiles y los fuertes.

**3. La feromona inicial.** El enunciado la fija a 1. Se deriva de una solución de referencia
la del vecino más cercano, para que el valor inicial esté en la escala de los depósitos que
vendrán después y las primeras hormigas no se guíen por ruido.

In [12]:
def solucion_vecino_mas_cercano(problem, inicio=0):
    """Recorrido construido yendo siempre a la ciudad no visitada más cercana."""
    Nodos = list(problem.get_nodes())
    camino = [inicio]
    pendientes = set(Nodos) - {inicio}

    while pendientes:
        actual = camino[-1]
        siguiente = min(pendientes, key=lambda c: problem.get_weight(actual, c))
        camino.append(siguiente)
        pendientes.discard(siguiente)

    return camino


def Add_Nodo(problem, H, T, alfa=1.0, beta=3.0):
    """Elige el siguiente nodo con la probabilidad del enunciado.

    p(i->j) proporcional a [feromona]^alfa * [visibilidad]^beta, con visibilidad = 1/distancia.
    La elección se hace por ruleta sobre los nodos aún no visitados.
    """
    actual = H[-1]
    visitados = set(H)
    candidatos = [c for c in range(len(T)) if c not in visitados]

    pesos = []
    for j in candidatos:
        d = problem.get_weight(actual, j)
        visibilidad = 1.0 / d if d > 0 else 1e6
        pesos.append((T[actual][j] ** alfa) * (visibilidad ** beta))

    suma = sum(pesos)
    if suma <= 0:
        return random.choice(candidatos)

    # Ruleta: se avanza acumulando pesos hasta pasar el umbral sorteado
    umbral = random.random() * suma
    acumulado = 0.0
    for j, peso in zip(candidatos, pesos):
        acumulado += peso
        if acumulado >= umbral:
            return j

    return candidatos[-1]


def Incrementa_Feromona(problem, T, H, Q=1000.0):
    """Deposita feromona inversamente proporcional a la longitud del recorrido."""
    aporte = Q / distancia_total(H, problem)
    for i in range(len(H)):
        a, b = H[i], H[(i + 1) % len(H)]     # El % cierra el ciclo: la última arista también cuenta
        T[a][b] += aporte
        T[b][a] += aporte                    # La matriz es simétrica
    return T


def Evaporar_Feromonas(T, rho=0.1, minimo=0.01):
    """Evaporación multiplicativa: tau <- (1-rho)*tau, con un suelo mínimo."""
    n = len(T)
    return [[max(T[i][j] * (1 - rho), minimo) for j in range(n)] for i in range(n)]


def feromona_inicial(problem, Q=1000.0):
    """Valor inicial derivado de una solución de referencia, no fijado a 1."""
    referencia = distancia_total(solucion_vecino_mas_cercano(problem), problem)
    return Q / referencia


def hormigas(problem, N=20, iteraciones=40, alfa=1.0, beta=3.0, rho=0.1):
    """Colonia de hormigas sobre el TSP.

    N = hormigas por iteración. En cada iteración cada hormiga construye un recorrido
    completo; al final de la iteración se deposita feromona y se evapora.
    """
    Nodos = list(problem.get_nodes())
    n = len(Nodos)

    tau0 = feromona_inicial(problem)
    T = [[tau0 for _ in range(n)] for _ in range(n)]

    mejor_solucion, mejor_distancia = None, float("inf")

    for _ in range(iteraciones):
        recorridos = []

        for _ in range(N):
            Hormiga = [Nodos[0]]                       # Todas parten de la ciudad 0
            for _ in range(n - 1):
                Hormiga.append(Add_Nodo(problem, Hormiga, T, alfa, beta))
            recorridos.append(Hormiga)

            d = distancia_total(Hormiga, problem)
            if d < mejor_distancia:
                mejor_solucion, mejor_distancia = Hormiga, d

        T = Evaporar_Feromonas(T, rho)                 # Primero se evapora...
        for Hormiga in recorridos:
            T = Incrementa_Feromona(problem, T, Hormiga)   # ...y luego se refuerza

    return mejor_solucion, mejor_distancia


random.seed(42)
sol_hormigas, dist_hormigas = hormigas(problem)
assert es_solucion_valida(sol_hormigas, problem)
print("Mejor solución:", sol_hormigas)
print("Distancia     :", dist_hormigas)

Mejor solución: [0, 1, 6, 4, 3, 2, 27, 28, 29, 30, 38, 22, 39, 21, 40, 24, 9, 23, 41, 8, 10, 25, 11, 12, 18, 26, 5, 13, 19, 14, 16, 15, 37, 7, 17, 31, 36, 35, 20, 34, 33, 32]
Distancia     : 1307


In [13]:
# ¿Influyen de verdad las feromonas y la visibilidad?
# Con alfa=0 y beta=0 todos los pesos valen 1: la ruleta pasa a ser uniforme y el
# algoritmo degenera exactamente en la construcción al azar del enunciado.
random.seed(42)
_, dist_ciega = hormigas(problem, N=20, iteraciones=40, alfa=0.0, beta=0.0)

print(f"Con alfa=1, beta=3 (feromona y visibilidad) : {dist_hormigas}")
print(f"Con alfa=0, beta=0 (elección uniforme)      : {dist_ciega}")
print(f"\nMejora al usar la fórmula de probabilidad   : {dist_ciega - dist_hormigas} "
      f"({(1 - dist_hormigas / dist_ciega) * 100:.0f} %)")

assert dist_hormigas < dist_ciega, "la fórmula de probabilidad debería mejorar la elección uniforme"

Con alfa=1, beta=3 (feromona y visibilidad) : 1307
Con alfa=0, beta=0 (elección uniforme)      : 3813

Mejora al usar la fórmula de probabilidad   : 2506 (66 %)


---
## 7. Comparativa y conclusiones

Las cuatro técnicas sobre la misma instancia. Como referencia, el **óptimo conocido de `swiss42` es
1273**: permite medir no solo cuál gana, sino a qué distancia se queda cada una del mejor recorrido
posible.

In [14]:
OPTIMO_SWISS42 = 1273    # Valor publicado en TSPLIB para esta instancia

resultados = []

random.seed(1)
inicio = time.perf_counter()
_, d, _ = busqueda_aleatoria(problem, N=10000)
resultados.append(("Búsqueda aleatoria", d, time.perf_counter() - inicio))

random.seed(1)
inicio = time.perf_counter()
_, d, _ = busqueda_local(problem, vecina_intercambio)
resultados.append(("Búsqueda local (intercambio)", d, time.perf_counter() - inicio))

random.seed(1)
inicio = time.perf_counter()
_, d, _ = busqueda_local(problem, vecina_2opt)
resultados.append(("Búsqueda local (2-opt)", d, time.perf_counter() - inicio))

random.seed(1)
inicio = time.perf_counter()
_, d, _ = recocido_simulado(problem)
resultados.append(("Recocido simulado", d, time.perf_counter() - inicio))

random.seed(1)
inicio = time.perf_counter()
_, d = hormigas(problem)
resultados.append(("Colonia de hormigas", d, time.perf_counter() - inicio))

print(f"{'técnica':<30} | {'distancia':>10} | {'sobre el óptimo':>16} | {'segundos':>9}")
print("-" * 74)
for nombre, distancia_obtenida, segundos in resultados:
    exceso = (distancia_obtenida / OPTIMO_SWISS42 - 1) * 100
    print(f"{nombre:<30} | {distancia_obtenida:>10} | {exceso:>15.1f}% | {segundos:>9.2f}")
print("-" * 74)
print(f"{'Óptimo conocido (TSPLIB)':<30} | {OPTIMO_SWISS42:>10} | {0.0:>15.1f}% |")

# Todas deben batir con claridad a la búsqueda aleatoria
distancia_aleatoria_ref = resultados[0][1]
for nombre, distancia_obtenida, _ in resultados[1:]:
    assert distancia_obtenida < distancia_aleatoria_ref, f"{nombre} no bate a la búsqueda aleatoria"

print("\nVerificado: las tres técnicas informadas baten a la búsqueda aleatoria.")

técnica                        |  distancia |  sobre el óptimo |  segundos
--------------------------------------------------------------------------
Búsqueda aleatoria             |       3784 |           197.3% |      0.27
Búsqueda local (intercambio)   |       1683 |            32.2% |      0.29
Búsqueda local (2-opt)         |       1331 |             4.6% |      0.40
Recocido simulado              |       1295 |             1.7% |      0.79
Colonia de hormigas            |       1372 |             7.8% |      0.57
--------------------------------------------------------------------------
Óptimo conocido (TSPLIB)       |       1273 |             0.0% |

Verificado: las tres técnicas informadas baten a la búsqueda aleatoria.


### Conclusiones

| Técnica | Qué usa del problema | Escapa de óptimos locales | Coste |
|---|---|---|---|
| Búsqueda aleatoria | Nada | No los busca siquiera | Muy bajo por muestra |
| Búsqueda local | La vecindad de la solución actual | **No** | Alto por iteración, pocas iteraciones |
| Recocido simulado | La vecindad, más la temperatura | **Sí**, aceptando empeoramientos | Muchas iteraciones baratas |
| Colonia de hormigas | Distancias y memoria compartida | Parcialmente, por la aleatoriedad | Alto: N recorridos por iteración |

**Lo que se aprende comparándolas:**

1. **La búsqueda aleatoria no escala.** Multiplicar por diez las muestras apenas mueve el
   resultado. En un espacio de $1{,}7\times10^{49}$ recorridos, muestrear sin criterio es
   inservible salvo como referencia inferior.

2. **El operador de vecindad pesa más que la metaheurística.** Es el resultado más claro de todo el
   trabajo: con el mismo esfuerzo, el mismo número de vecinas y los mismos puntos de partida,
   cambiar el intercambio de ciudades por la inversión de segmento mejora más que pasar de búsqueda
   local a recocido simulado. Antes de buscar un algoritmo más sofisticado conviene revisar cómo se
   define «vecina».

3. **El recocido compra escapatoria con evaluaciones.** Acepta empeoramientos y por eso sale de los
   óptimos locales donde la búsqueda local se queda encerrada, pero necesita decenas de miles de
   evaluaciones frente a las pocas decenas de iteraciones de aquélla. Y depende de la calibración:
   con la temperatura mal escogida degenera en un paseo aleatorio.

4. **En la colonia de hormigas, el mecanismo está en la probabilidad.** Depositar y evaporar
   feromona no sirve de nada si la elección del siguiente nodo no la mira: sin la fórmula
   $[\tau]^\alpha[\nu]^\beta$, el algoritmo es búsqueda aleatoria con pasos de más. La comparación
   con $\alpha=\beta=0$ lo mide directamente.

5. **Ninguna garantiza el óptimo.** A diferencia de la programación dinámica o la ramificación y
   poda de la AG2, estas técnicas no demuestran nada: devuelven la mejor solución que han sabido
   encontrar. Por eso disponer de una referencia aquí, el óptimo conocido 1273 es lo que permite
   juzgar si un resultado es bueno o solo es el mejor que hemos visto.

---
## 8. Uso de inteligencia artificial

La inteligencia artificial fue utilizada como herramienta de apoyo en la redacción del presente trabajo, principalmente para mejorar la claridad de las ideas, corregir aspectos de redacción y evitar la repetición o redundancia de palabras y expresiones. El contenido, análisis y conclusiones presentados son responsabilidad del autor.